# Transient Stokes Flow Around a Cylinder — Documentation

This notebook documents the transient Stokes flow tutorial. It follows the same section-by-section style as the reference notebook: each block first explains the math, then shows the Julia code that implements it. The last cell contains the complete runnable script so the notebook can stand on its own.

## 1. Governing equations

Transient Stokes flow is the low-Reynolds-number limit of Navier-Stokes, where the nonlinear convective term is neglected. The strong form is

$$
\frac{\partial \mathbf{u}}{\partial t} - \nu \nabla^2 \mathbf{u} + \nabla p = \mathbf{f},
$$

$$
\nabla \cdot \mathbf{u} = 0.
$$

After testing with velocity test functions $\mathbf{v}$ and pressure test functions $q$, the weak form becomes

$$
\int_\Omega \frac{\partial \mathbf{u}}{\partial t} \cdot \mathbf{v}\, d\Omega
+ \nu \int_\Omega \nabla \mathbf{u} : \nabla \mathbf{v}\, d\Omega
- \int_\Omega p\, \nabla \cdot \mathbf{v}\, d\Omega = 0,
$$

$$
\int_\Omega q\, \nabla \cdot \mathbf{u}\, d\Omega = 0.
$$

Discretizing in space gives the DAE system $\mathbf{M}\dot{\mathbf{U}} = \mathbf{K}\mathbf{U}$, where $\mathbf{M}$ is the velocity mass matrix and $\mathbf{K}$ is the Stokes operator.

### Imports and parameters

The code uses Ferrite for finite elements, FerriteGmsh for the mesh, and OrdinaryDiffEqRosenbrock for time integration. The fluid viscosity is the only physical parameter in this linear model.

In [ ]:
using Ferrite, SparseArrays, BlockArrays, LinearAlgebra, WriteVTK
using DiffEqBase
using OrdinaryDiffEqRosenbrock: Rodas5P

ν = 1.0 / 1000.0 # kinematic viscosity

## 2. Geometry and mesh generation

The geometry is the same channel-with-cylinder setup used in the Navier-Stokes tutorial. A rectangle defines the outer channel and a circular obstacle is subtracted from it using a boolean cut in Gmsh. Physical boundary groups are then assigned to the walls, inlet, outlet, and cylinder.

In [ ]:
using FerriteGmsh
using FerriteGmsh: Gmsh
Gmsh.initialize()
gmsh.option.set_number("General.Verbosity", 2)
dim = 2

rect_tag = gmsh.model.occ.add_rectangle(0, 0, 0, 1.1, 0.41)
circle_tag = gmsh.model.occ.add_circle(0.2, 0.2, 0, 0.05)
circle_curve_tag = gmsh.model.occ.add_curve_loop([circle_tag])
circle_surf_tag = gmsh.model.occ.add_plane_surface([circle_curve_tag])
gmsh.model.occ.cut([(dim, rect_tag)], [(dim, circle_surf_tag)])
gmsh.model.occ.synchronize()

bottomtag = gmsh.model.model.add_physical_group(dim - 1, [6], -1, "bottom")
lefttag   = gmsh.model.model.add_physical_group(dim - 1, [7], -1, "left")
righttag  = gmsh.model.model.add_physical_group(dim - 1, [8], -1, "right")
toptag    = gmsh.model.model.add_physical_group(dim - 1, [9], -1, "top")
holetag   = gmsh.model.model.add_physical_group(dim - 1, [5], -1, "hole")

gmsh.option.setNumber("Mesh.Algorithm", 11)
gmsh.option.setNumber("Mesh.MeshSizeFromCurvature", 20)
gmsh.option.setNumber("Mesh.MeshSizeMax", 0.05)

gmsh.model.mesh.generate(dim)
grid = togrid()
Gmsh.finalize()

## 3. Finite element spaces

We again use Taylor-Hood elements: quadratic velocity and linear pressure. This gives a stable mixed discretization for incompressible flow.

In [ ]:
ip_v = Lagrange{RefQuadrilateral, 2}()^dim
qr   = QuadratureRule{RefQuadrilateral}(4)
cellvalues_v = CellValues(qr, ip_v)

ip_p = Lagrange{RefQuadrilateral, 1}()
cellvalues_p = CellValues(qr, ip_p)

dh = DofHandler(grid)
add!(dh, :v, ip_v)
add!(dh, :p, ip_p)
close!(dh)

## 4. Boundary conditions

The boundary conditions match the Navier-Stokes tutorial:

$$
\mathbf{u} = \mathbf{0} \qquad \text{on the walls and cylinder},
$$

$$
\mathbf{u}(0,y,t) = \left(u_{\text{in}}(t)\frac{4y(H-y)}{H^2}, 0\right)^T \qquad \text{on the inlet},
$$

with the time-ramped inflow amplitude $u_{	ext{in}}(t)=\min(t v_{\max}, v_{\max})$. The outlet is left free.

In [ ]:
ch = ConstraintHandler(dh)

noslip_facet_names = ["top", "bottom", "hole"]
∂Ω_noslip = union(getfacetset.((grid,), noslip_facet_names)...)
noslip_bc = Dirichlet(:v, ∂Ω_noslip, (x, t) -> Vec((0.0, 0.0)), [1, 2])
add!(ch, noslip_bc)

∂Ω_inflow = getfacetset(grid, "left")
vᵢₙ(t) = min(t * 1.5, 1.5)
parabolic_inflow_profile(x, t) = Vec((4 * vᵢₙ(t) * x[2] * (0.41 - x[2]) / 0.41^2, 0.0))
inflow_bc = Dirichlet(:v, ∂Ω_inflow, parabolic_inflow_profile, [1, 2])
add!(ch, inflow_bc)

∂Ω_free = getfacetset(grid, "right")
close!(ch)
update!(ch, 0.0)

## 5. Mass matrix

The transient term contributes the velocity mass matrix

$$
\int_\Omega \frac{\partial \mathbf{u}}{\partial t} \cdot \mathbf{v}\, d\Omega.
$$

Because pressure has no time derivative, only the velocity-velocity block is non-zero.

In [ ]:
function assemble_mass_matrix(cellvalues_v::CellValues, cellvalues_p::CellValues, M::SparseMatrixCSC, dh::DofHandler)
    n_basefuncs_v = getnbasefunctions(cellvalues_v)
    n_basefuncs_p = getnbasefunctions(cellvalues_p)
    n_basefuncs   = n_basefuncs_v + n_basefuncs_p
    v▄, p▄ = 1, 2
    Mₑ = BlockedArray(zeros(n_basefuncs, n_basefuncs), [n_basefuncs_v, n_basefuncs_p], [n_basefuncs_v, n_basefuncs_p])

    mass_assembler = start_assemble(M)
    for cell in CellIterator(dh)
        fill!(Mₑ, 0)
        Ferrite.reinit!(cellvalues_v, cell)
        for q_point in 1:getnquadpoints(cellvalues_v)
            dΩ = getdetJdV(cellvalues_v, q_point)
            for i in 1:n_basefuncs_v
                φᵢ = shape_value(cellvalues_v, q_point, i)
                for j in 1:n_basefuncs_v
                    φⱼ = shape_value(cellvalues_v, q_point, j)
                    Mₑ[BlockIndex((v▄, v▄), (i, j))] += φᵢ ⋅ φⱼ * dΩ
                end
            end
        end
        assemble!(mass_assembler, celldofs(cell), Mₑ)
    end
    return M
end

## 6. Stokes stiffness matrix

The viscous block is

$$
-\nu \int_\Omega \nabla \phi_i : \nabla \phi_j\, d\Omega,
$$

and the pressure coupling blocks are

$$
\int_\Omega (\nabla \cdot \phi_i)\psi_j\, d\Omega \qquad \text{and} \qquad \int_\Omega \psi_i (\nabla \cdot \phi_j)\, d\Omega.
$$

This gives the saddle-point operator for the incompressible Stokes equations.

In [ ]:
function assemble_stokes_matrix(cellvalues_v::CellValues, cellvalues_p::CellValues, ν, K::SparseMatrixCSC, dh::DofHandler)
    n_basefuncs_v = getnbasefunctions(cellvalues_v)
    n_basefuncs_p = getnbasefunctions(cellvalues_p)
    n_basefuncs   = n_basefuncs_v + n_basefuncs_p
    v▄, p▄ = 1, 2
    Kₑ = BlockedArray(zeros(n_basefuncs, n_basefuncs), [n_basefuncs_v, n_basefuncs_p], [n_basefuncs_v, n_basefuncs_p])

    stiffness_assembler = start_assemble(K)
    for cell in CellIterator(dh)
        fill!(Kₑ, 0)
        Ferrite.reinit!(cellvalues_v, cell)
        Ferrite.reinit!(cellvalues_p, cell)
        for q_point in 1:getnquadpoints(cellvalues_v)
            dΩ = getdetJdV(cellvalues_v, q_point)
            for i in 1:n_basefuncs_v
                ∇φᵢ = shape_gradient(cellvalues_v, q_point, i)
                for j in 1:n_basefuncs_v
                    ∇φⱼ = shape_gradient(cellvalues_v, q_point, j)
                    Kₑ[BlockIndex((v▄, v▄), (i, j))] -= ν * ∇φᵢ ⊡ ∇φⱼ * dΩ
                end
            end
            for j in 1:n_basefuncs_p
                ψ = shape_value(cellvalues_p, q_point, j)
                for i in 1:n_basefuncs_v
                    divφ = shape_divergence(cellvalues_v, q_point, i)
                    Kₑ[BlockIndex((v▄, p▄), (i, j))] += (divφ * ψ) * dΩ
                    Kₑ[BlockIndex((p▄, v▄), (j, i))] += (ψ * divφ) * dΩ
                end
            end
        end
        assemble!(stiffness_assembler, celldofs(cell), Kₑ)
    end
    return K
end

## 7. Time integration and residual

Because the Stokes problem is linear, the right-hand side is simply $\mathbf{K}\mathbf{u}$. The Jacobian is equal to $\mathbf{K}$. The code uses a limiter to keep the constrained degrees of freedom fixed after each step and integrates the DAE with `Rodas5P`.

In [ ]:
struct RHSparams
    K::SparseMatrixCSC
    ch::ConstraintHandler
    dh::DofHandler
    cellvalues_v::CellValues
    u::Vector
end

T      = 20.0
Δt₀    = 0.001
Δt_save = 0.1

M = allocate_matrix(dh)
M = assemble_mass_matrix(cellvalues_v, cellvalues_p, M, dh)
K = allocate_matrix(dh)
K = assemble_stokes_matrix(cellvalues_v, cellvalues_p, ν, K, dh)
u₀ = zeros(ndofs(dh))
apply!(u₀, ch)
jac_sparsity = sparse(K)
apply!(M, ch)
p = RHSparams(K, ch, dh, cellvalues_v, copy(u₀))

function ferrite_limiter!(u, _, p, t)
    update!(p.ch, t)
    return apply!(u, p.ch)
end

function stokes!(du, u_uc, p::RHSparams, t)
    (; K, ch, u) = p
    u .= u_uc
    update!(ch, t)
    apply!(u, ch)
    mul!(du, K, u)
    return
end

function stokes_jac!(J, u_uc, p::RHSparams, t)
    (; K, ch, u) = p
    u .= u_uc
    update!(ch, t)
    apply!(u, ch)
    nonzeros(J) .= nonzeros(K)
    return apply!(J, ch)
end

struct FreeDofErrorNorm
    ch::ConstraintHandler
end
(fe_norm::FreeDofErrorNorm)(u::Union{AbstractFloat, Complex}, t) = DiffEqBase.ODE_DEFAULT_NORM(u, t)
(fe_norm::FreeDofErrorNorm)(u::AbstractArray, t) = DiffEqBase.ODE_DEFAULT_NORM(u[fe_norm.ch.free_dofs], t)

rhs = ODEFunction(stokes!, mass_matrix = M; jac = stokes_jac!, jac_prototype = jac_sparsity)
problem = ODEProblem(rhs, u₀, (0.0, T), p)
timestepper = Rodas5P(autodiff = false, step_limiter! = ferrite_limiter!)
integrator = init(
    problem, timestepper; initializealg = NoInit(), dt = Δt₀,
    adaptive = true, abstol = 1.0e-4, reltol = 1.0e-5,
    progress = true, progress_steps = 1,
    verbose = true, internalnorm = FreeDofErrorNorm(ch), d_discontinuities = [1.0]
)

## 8. Output and visualization

The time evolution is written to VTU/PVD files so the solution can be opened in ParaView. Each saved time step stores the velocity and pressure fields on the finite element mesh.

In [ ]:
pvd = paraview_collection("stokes_trans_cyl_20")
for (step, (u, t)) in enumerate(intervals(integrator))
    VTKGridFile("stokes_trans_cyl_20-$step", dh) do vtk
        write_solution(vtk, dh, u)
        pvd[t] = vtk
    end
end
vtk_save(pvd)
println("Done!")

## 9. Full runnable script

The cell below contains the full original transient Stokes tutorial so the notebook can be executed from top to bottom without the source `.jl` file.

In [ ]:
# Transient Stokes flow around a cylinder
# Based on the Navier-Stokes tutorial (ns_tut_cyl.jl) with the convective term dropped.
# The governing equation is:
#   M * du/dt = K * u
# where K is the Stokes operator (viscous diffusion + pressure-velocity coupling)
# and M is the velocity mass matrix.

using Ferrite, SparseArrays, BlockArrays, LinearAlgebra, WriteVTK

using DiffEqBase
using OrdinaryDiffEqRosenbrock: Rodas5P

ν = 1.0 / 1000.0 # kinematic viscosity

using FerriteGmsh
using FerriteGmsh: Gmsh
Gmsh.initialize()
gmsh.option.set_number("General.Verbosity", 2)
dim = 2

# Build the channel geometry with a circular obstacle
rect_tag = gmsh.model.occ.add_rectangle(0, 0, 0, 1.1, 0.41)
circle_tag = gmsh.model.occ.add_circle(0.2, 0.2, 0, 0.05)
circle_curve_tag = gmsh.model.occ.add_curve_loop([circle_tag])
circle_surf_tag = gmsh.model.occ.add_plane_surface([circle_curve_tag])
gmsh.model.occ.cut([(dim, rect_tag)], [(dim, circle_surf_tag)])

gmsh.model.occ.synchronize()

bottomtag = gmsh.model.model.add_physical_group(dim - 1, [6], -1, "bottom")
lefttag   = gmsh.model.model.add_physical_group(dim - 1, [7], -1, "left")
righttag  = gmsh.model.model.add_physical_group(dim - 1, [8], -1, "right")
toptag    = gmsh.model.model.add_physical_group(dim - 1, [9], -1, "top")
holetag   = gmsh.model.model.add_physical_group(dim - 1, [5], -1, "hole")

gmsh.option.setNumber("Mesh.Algorithm", 11)
gmsh.option.setNumber("Mesh.MeshSizeFromCurvature", 20)
gmsh.option.setNumber("Mesh.MeshSizeMax", 0.05)

gmsh.model.mesh.generate(dim)
grid = togrid()
Gmsh.finalize()

# Taylor-Hood Q2/Q1 elements (same as NS tutorial)
ip_v = Lagrange{RefQuadrilateral, 2}()^dim
qr   = QuadratureRule{RefQuadrilateral}(4)
cellvalues_v = CellValues(qr, ip_v)

ip_p = Lagrange{RefQuadrilateral, 1}()
cellvalues_p = CellValues(qr, ip_p)

dh = DofHandler(grid)
add!(dh, :v, ip_v)
add!(dh, :p, ip_p)
close!(dh)

# Boundary conditions (identical to NS tutorial)
ch = ConstraintHandler(dh)

noslip_facet_names = ["top", "bottom", "hole"]
∂Ω_noslip = union(getfacetset.((grid,), noslip_facet_names)...)
noslip_bc = Dirichlet(:v, ∂Ω_noslip, (x, t) -> Vec((0.0, 0.0)), [1, 2])
add!(ch, noslip_bc)

∂Ω_inflow = getfacetset(grid, "left")
vᵢₙ(t) = min(t * 1.5, 1.5)
parabolic_inflow_profile(x, t) = Vec((4 * vᵢₙ(t) * x[2] * (0.41 - x[2]) / 0.41^2, 0.0))
inflow_bc = Dirichlet(:v, ∂Ω_inflow, parabolic_inflow_profile, [1, 2])
add!(ch, inflow_bc)

∂Ω_free = getfacetset(grid, "right")

close!(ch)
update!(ch, 0.0)

# Assemble the velocity mass matrix M
function assemble_mass_matrix(cellvalues_v::CellValues, cellvalues_p::CellValues, M::SparseMatrixCSC, dh::DofHandler)
    n_basefuncs_v = getnbasefunctions(cellvalues_v)
    n_basefuncs_p = getnbasefunctions(cellvalues_p)
    n_basefuncs   = n_basefuncs_v + n_basefuncs_p
    v▄, p▄ = 1, 2
    Mₑ = BlockedArray(zeros(n_basefuncs, n_basefuncs),
                      [n_basefuncs_v, n_basefuncs_p],
                      [n_basefuncs_v, n_basefuncs_p])

    mass_assembler = start_assemble(M)
    for cell in CellIterator(dh)
        fill!(Mₑ, 0)
        Ferrite.reinit!(cellvalues_v, cell)
        for q_point in 1:getnquadpoints(cellvalues_v)
            dΩ = getdetJdV(cellvalues_v, q_point)
            for i in 1:n_basefuncs_v
                φᵢ = shape_value(cellvalues_v, q_point, i)
                for j in 1:n_basefuncs_v
                    φⱼ = shape_value(cellvalues_v, q_point, j)
                    Mₑ[BlockIndex((v▄, v▄), (i, j))] += φᵢ ⋅ φⱼ * dΩ
                end
            end
        end
        assemble!(mass_assembler, celldofs(cell), Mₑ)
    end
    return M
end

# Assemble the Stokes stiffness matrix K
function assemble_stokes_matrix(cellvalues_v::CellValues, cellvalues_p::CellValues, ν, K::SparseMatrixCSC, dh::DofHandler)
    n_basefuncs_v = getnbasefunctions(cellvalues_v)
    n_basefuncs_p = getnbasefunctions(cellvalues_p)
    n_basefuncs   = n_basefuncs_v + n_basefuncs_p
    v▄, p▄ = 1, 2
    Kₑ = BlockedArray(zeros(n_basefuncs, n_basefuncs),
                      [n_basefuncs_v, n_basefuncs_p],
                      [n_basefuncs_v, n_basefuncs_p])

    stiffness_assembler = start_assemble(K)
    for cell in CellIterator(dh)
        fill!(Kₑ, 0)
        Ferrite.reinit!(cellvalues_v, cell)
        Ferrite.reinit!(cellvalues_p, cell)
        for q_point in 1:getnquadpoints(cellvalues_v)
            dΩ = getdetJdV(cellvalues_v, q_point)
            for i in 1:n_basefuncs_v
                ∇φᵢ = shape_gradient(cellvalues_v, q_point, i)
                for j in 1:n_basefuncs_v
                    ∇φⱼ = shape_gradient(cellvalues_v, q_point, j)
                    Kₑ[BlockIndex((v▄, v▄), (i, j))] -= ν * ∇φᵢ ⊡ ∇φⱼ * dΩ
                end
            end
            for j in 1:n_basefuncs_p
                ψ = shape_value(cellvalues_p, q_point, j)
                for i in 1:n_basefuncs_v
                    divφ = shape_divergence(cellvalues_v, q_point, i)
                    Kₑ[BlockIndex((v▄, p▄), (i, j))] += (divφ * ψ) * dΩ
                    Kₑ[BlockIndex((p▄, v▄), (j, i))] += (ψ * divφ) * dΩ
                end
            end
        end
        assemble!(stiffness_assembler, celldofs(cell), Kₑ)
    end
    return K
end

# Time integration parameters
T      = 20.0
Δt₀    = 0.001
Δt_save = 0.1

M = allocate_matrix(dh)
M = assemble_mass_matrix(cellvalues_v, cellvalues_p, M, dh)
K = allocate_matrix(dh)
K = assemble_stokes_matrix(cellvalues_v, cellvalues_p, ν, K, dh)
u₀ = zeros(ndofs(dh))
apply!(u₀, ch)
jac_sparsity = sparse(K)
apply!(M, ch)

struct RHSparams
    K::SparseMatrixCSC
    ch::ConstraintHandler
    dh::DofHandler
    cellvalues_v::CellValues
    u::Vector
end
p = RHSparams(K, ch, dh, cellvalues_v, copy(u₀))

function ferrite_limiter!(u, _, p, t)
    update!(p.ch, t)
    return apply!(u, p.ch)
end

function stokes!(du, u_uc, p::RHSparams, t)
    (; K, ch, u) = p
    u .= u_uc
    update!(ch, t)
    apply!(u, ch)
    mul!(du, K, u)
    return
end

function stokes_jac!(J, u_uc, p::RHSparams, t)
    (; K, ch, u) = p
    u .= u_uc
    update!(ch, t)
    apply!(u, ch)
    nonzeros(J) .= nonzeros(K)
    return apply!(J, ch)
end

struct FreeDofErrorNorm
    ch::ConstraintHandler
end
(fe_norm::FreeDofErrorNorm)(u::Union{AbstractFloat, Complex}, t) = DiffEqBase.ODE_DEFAULT_NORM(u, t)
(fe_norm::FreeDofErrorNorm)(u::AbstractArray, t) = DiffEqBase.ODE_DEFAULT_NORM(u[fe_norm.ch.free_dofs], t)

rhs = ODEFunction(stokes!, mass_matrix = M; jac = stokes_jac!, jac_prototype = jac_sparsity)
problem = ODEProblem(rhs, u₀, (0.0, T), p)
timestepper = Rodas5P(autodiff = false, step_limiter! = ferrite_limiter!)
integrator = init(
    problem, timestepper; initializealg = NoInit(), dt = Δt₀,
    adaptive = true, abstol = 1.0e-4, reltol = 1.0e-5,
    progress = true, progress_steps = 1,
    verbose = true, internalnorm = FreeDofErrorNorm(ch), d_discontinuities = [1.0]
)

pvd = paraview_collection("stokes_trans_cyl_20")
for (step, (u, t)) in enumerate(intervals(integrator))
    VTKGridFile("stokes_trans_cyl_20-$step", dh) do vtk
        write_solution(vtk, dh, u)
        pvd[t] = vtk
    end
end
vtk_save(pvd)
println("Done!")

---

Notes:
- Run the notebook top-to-bottom in a Julia kernel (IJulia).
- The full script cell duplicates the sectioned explanation so the notebook can run standalone.
- The outlet is left free, matching the original transient Stokes tutorial.